In [ ]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','srchResultListW').find_all('li')
for i in content_1 :
    print(i.get_text().replace("\n",""))

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)
print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ] #번호 저장
title2 = [ ] #논문제목 저장
writer2 = [ ] #논문저자 저장
org2 = [ ] #소속기관 저장
no = 1

for a in range(1, collect_page_cnt + 1) :
    
    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')
    
    for b in content_2 :
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text()
        except :
            continue # 논문제목 못찾으면 오류 발생, 해당 항목 건너뜀 
        else :
            f = open(ft_name, 'a' , encoding="UTF-8") #append(추가모드), 기존 파일에 계속 추가 
            print('1.번호:',no)
            no2.append(no)
            f.write('\n'+'1.번호:' + str(no))

            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n' + '2.논문제목:' + title)

            writer = b.find('span','writer').get_text()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n' + '3.저자:' + writer)

            org = b.find('span','assigned').get_text()
            print('4.소속기관:' , org)
            org2.append(org)
            f.write('\n' + '4.소속기관:' + org + '\n')

            f.close( )

            no += 1
            print("\n")

            
            if no >= collect_cnt :
                break


            

    time.sleep(1) # 페이지 변경 전 1초 대기

  
    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
    except :
        driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        print("요청하신 작업이 모두 완료되었습니다")
    time.sleep(1) # 페이지 변경 전 1초 대기



# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
import pandas as pd

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(writer2)
df['소속(발행)기관']=pd.Series(org2)

# xls 형태로 저장하기
df.to_excel(fx_name, index=False, encoding="utf-8" , engine='openpyxl')

# csv 형태로 저장하기
df.to_csv(fc_name, index=False, encoding="utf-8-sig")
print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
1해양관광유형과 관광자원 가치가 주민참여의도에 미치는 영향 : 당진시를 중심으로이길호경기대학교 관광전문대학원2014국내박사RANK : 13880191원문보기목차검색조회음성듣기21세기는 문화시대이자 창의적 산업시대이며 감성적 소통 시대이다. 최근 들어 국민소득 증대와 함께 생활수준이 향상되고 주 5일 근무 정착에 따른 여가시간 확대로 관광 및 레저에 대한 관심도가 증가하고 있으며 관광 패턴이 육지위주에서 해양위주로 변화하고 있다. 3면이 바다로 둘러 싸여있는 우리나라는 천혜의 해양관광자원 보고이며 축복받은 해양관광자원이 곳곳에 산재되어 있다. 따라서 해양관광자원을 개발위주의 근대 공간에서 탈피하고 장소성이 있는 공간으로 창출하여 관광자와 지역주민의 삶의 질을 높이는 한편, 자연과 환경을 보전하고 지역주민 경제적 활성화를 증대 시키는 일이 요구되고 있다. 따라서 지방자치제도 시행 이후 각 지자체들은 지역경제 활성화 방안으로 지역종합발전계획을 수립하고 더 많은 외래관광객 유입을 목표로 관광시책을 추진하고 있으나 지자체 관심과 의욕에 비하여 관광발전시책이 성공적이지 못한 사례가 여러 곳에서 나타나고 있는 현실이다. 그 원인의 대부분은 관광객 유인을 위한 방안으로 관주도의 일방적인 관광개발에 치중하여 새로운 관광지 조성을 위한 개발 사업에 주력해 왔기 때문이다. 이는 공급자 중심 시설개발에서 수요자 중심 관광만족도 제고를 위한 방법이 미흡한 것으로 해석된다. 따라서 관주도 관광자원 관리방식을 탈피하여 지자체와 관광전문가, 관광사업자, 그리고 지역주민과 협의 하에 보호, 보전, 개발의 관광자원 관리의 필요성이 중요시 되고 있다.우리나라는 육지면적의 3배가 넘는 34만 5,000㎢에 달하는 대륙붕과 수심 20m 내․외 해역도 국토 3분의 1에 해당하며, 11,542㎞에 달하는 해안선을 보유하고 있다. 또한 3,200여 개의 도서를 보유하고 있으며, 갯벌 면적은 남한 면적 2.5%에 달하는 2,815㎢

ModuleNotFoundError: No module named 'pandas'

In [19]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','srchResultListW').find_all('li')
for i in content_1 :
    print(i.get_text().replace("\n",""))

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)
print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ] #번호 저장
title2 = [ ] #논문제목 저장
writer2 = [ ] #논문저자 저장
org2 = [ ] #소속기관 저장
no = 1

for a in range(1, collect_page_cnt + 1) :
    
    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')
    
    for b in content_2 :
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text()
        except :
            continue # 논문제목 못찾으면 오류 발생, 해당 항목 건너뜀 
        else :
            f = open(ft_name, 'a' , encoding="UTF-8") #append(추가모드), 기존 파일에 계속 추가 
            print('1.번호:',no)
            no2.append(no)
            f.write('\n'+'1.번호:' + str(no))

            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n' + '2.논문제목:' + title)

            writer = b.find('span','writer').get_text()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n' + '3.저자:' + writer)

            org = b.find('span','assigned').get_text()
            print('4.소속기관:' , org)
            org2.append(org)
            f.write('\n' + '4.소속기관:' + org + '\n')

            f.close( )

            no += 1
            print("\n")

            
            if no > collect_cnt :
                break

            time.sleep(1) 

    a += 1
    b = str(a)
            

    time.sleep(1) # 페이지 변경 전 1초 대기

  
    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
    except :
        driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        print("요청하신 작업이 모두 완료되었습니다")
    time.sleep(1) # 페이지 변경 전 1초 대기



# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
import pandas as pd

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(writer2)
df['소속(발행)기관']=pd.Series(org2)

# xls 형태로 저장하기
df.to_excel(fx_name, index=False, encoding="utf-8" , engine='openpyxl')

# csv 형태로 저장하기
df.to_csv(fc_name, index=False, encoding="utf-8-sig")
print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.
1해양관광유형과 관광자원 가치가 주민참여의도에 미치는 영향 : 당진시를 중심으로이길호경기대학교 관광전문대학원2014국내박사RANK : 13880191원문보기목차검색조회음성듣기21세기는 문화시대이자 창의적 산업시대이며 감성적 소통 시대이다. 최근 들어 국민소득 증대와 함께 생활수준이 향상되고 주 5일 근무 정착에 따른 여가시간 확대로 관광 및 레저에 대한 관심도가 증가하고 있으며 관광 패턴이 육지위주에서 해양위주로 변화하고 있다. 3면이 바다로 둘러 싸여있는 우리나라는 천혜의 해양관광자원 보고이며 축복받은 해양관광자원이 곳곳에 산재되어 있다. 따라서 해양관광자원을 개발위주의 근대 공간에서 탈피하고 장소성이 있는 공간으로 창출하여 관광자와 지역주민의 삶의 질을 높이는 한편, 자연과 환경을 보전하고 지역주민 경제적 활성화를 증대 시키는 일이 요구되고 있다. 따라서 지방자치제도 시행 이후 각 지자체들은 지역경제 활성화 방안으로 지역종합발전계획을 수립하고 더 많은 외래관광객 유입을 목표로 관광시책을 추진하고 있으나 지자체 관심과 의욕에 비하여 관광발전시책이 성공적이지 못한 사례가 여러 곳에서 나타나고 있는 현실이다. 그 원인의 대부분은 관광객 유인을 위한 방안으로 관주도의 일방적인 관광개발에 치중하여 새로운 관광지 조성을 위한 개발 사업에 주력해 왔기 때문이다. 이는 공급자 중심 시설개발에서 수요자 중심 관광만족도 제고를 위한 방법이 미흡한 것으로 해석된다. 따라서 관주도 관광자원 관리방식을 탈피하여 지자체와 관광전문가, 관광사업자, 그리고 지역주민과 협의 하에 보호, 보전, 개발의 관광자원 관리의 필요성이 중요시 되고 있다.우리나라는 육지면적의 3배가 넘는 34만 5,000㎢에 달하는 대륙붕과 수심 20m 내․외 해역도 국토 3분의 1에 해당하며, 11,542㎞에 달하는 해안선을 보유하고 있다. 또한 3,200여 개의 도서를 보유하고 있으며, 갯벌 면적은 남한 면적 2.5%에 달하는 2,815㎢

TypeError: NDFrame.to_excel() got an unexpected keyword argument 'encoding'

In [ ]:
#pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------------------------ --------------- 6.0/9.7 MB 44.2 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 47.7 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 12.3/12.3 MB 80.1 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   -------------

이게 최종 코드 

In [ ]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','srchResultListW').find_all('li')
for i in content_1 :
    print(i.get_text().replace("\n",""))

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)
print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ] #번호 저장
title2 = [ ] #논문제목 저장
writer2 = [ ] #논문저자 저장
org2 = [ ] #소속기관 저장
no = 1



for a in range(1, collect_page_cnt + 1) :
    
    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')
    
    for b in content_2 :
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text()
        except :
            continue # 논문제목 못찾으면 오류 발생, 해당 항목 건너뜀 
        else :
            f = open(ft_name, 'a' , encoding="UTF-8") #append(추가모드), 기존 파일에 계속 추가 
            print('1.번호:',no)
            no2.append(no)
            f.write('\n'+'1.번호:' + str(no))

            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n' + '2.논문제목:' + title)

            writer = b.find('span','writer').get_text()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n' + '3.저자:' + writer)

            org = b.find('span','assigned').get_text()
            print('4.소속기관:' , org)
            org2.append(org)
            f.write('\n' + '4.소속기관:' + org + '\n')

            f.close( )

            no += 1
            print("\n")

            
            if no > collect_cnt :
                break

            time.sleep(1) 

    a += 1
    b = str(a)
            

    time.sleep(1) # 페이지 변경 전 1초 대기

  
    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
    except :
        driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        print("요청하신 작업이 모두 완료되었습니다")
    time.sleep(1) # 페이지 변경 전 1초 대기



# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
import pandas as pd

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(writer2)
df['소속(발행)기관']=pd.Series(org2)

# xls 형태로 저장하기
df.to_excel(fx_name, index=False,  engine='openpyxl')

#encoding="utf-8" ,

# csv 형태로 저장하기
df.to_csv(fc_name, index=False, encoding="utf-8-sig")
print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')